In [1]:
# Importar librerías básicas
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Algoritmos de clustering
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

# Librerías de evaluación
from sklearn.metrics import silhouette_score, silhouette_samples

# Configuraciones del notebook
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Cargar los datos
path = 'https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/Mall_Customers.csv'
df = pd.read_csv(path)

# Mostrar primeras filas
df.head()

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


# Exploratory Data Analysis

In [3]:
df.describe()

,CustomerID,Age,Annual Income (k$),Spending Score (1-100)
count,200.000000,200.000000,200.000000,200.000000
mean,100.500000,38.850000,60.560000,50.200000
std,57.879185,13.969007,26.264721,25.823522
min,1.000000,18.000000,15.000000,1.000000
25%,50.750000,28.750000,41.500000,34.750000
50%,100.500000,36.000000,61.500000,50.000000
75%,150.250000,49.000000,78.000000,73.000000
max,200.000000,70.000000,137.000000,99.000000


In [4]:
fig = px.scatter(
    df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    title='Gráfico de dispersión',
    # color='Genre'
)

fig.update_traces(marker=dict(size=10))
fig.show()

# DBSCAN

In [5]:
# Primero con 2 dimensiones
# Crear nuestra matriz X
X_2d = df.drop(columns=['CustomerID', 'Genre', 'Age'])

# Escalar nuestros datos
scaler = StandardScaler()
X_scaled_2d = scaler.fit_transform(X_2d)

X_scaled_2d.shape

(200, 2)

In [6]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=5,
    metric='euclidean'
)

dbscan.fit(X_scaled_2d)
dbscan

,"eps eps: float, default=0.5The maximum distance between two samples for one to be consideredas in the neighborhood of the other. This is not a maximum boundon the distances of points within a cluster. This is the mostimportant DBSCAN parameter to choose appropriately for your data setand distance function. Smaller values generally lead to more clusters.",0.5
,"min_samples min_samples: int, default=5The number of samples (or total weight) in a neighborhood for a point tobe considered as a core point. This includes the point itself. If`min_samples` is set to a higher value, DBSCAN will find denser clusters,whereas if it is set to a lower value, the found clusters will be moresparse.",5
,"metric metric: str, or callable, default='euclidean'The metric to use when calculating distance between instances in afeature array. If metric is a string or callable, it must be one ofthe options allowed by :func:`sklearn.metrics.pairwise_distances` forits metric parameter.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors for DBSCAN... versionadded:: 0.17 metric *precomputed* to accept precomputed sparse matrix.",'euclidean'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function... versionadded:: 0.19",None
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'The algorithm to be used by the NearestNeighbors moduleto compute pointwise distances and find nearest neighbors.'auto' will attempt to decide the most appropriate algorithmbased on the values passed to :meth:`fit` method.See :class:`~sklearn.neighbors.NearestNeighbors` documentation fordetails.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or cKDTree. This can affect the speedof the construction and query, as well as the memory requiredto store the tree. The optimal value dependson the nature of the problem.",30
,"p p: float, default=NoneThe power of the Minkowski metric to be used to calculate distancebetween points. If None, then ``p=2`` (equivalent to the Euclideandistance). When p=1, this is equivalent to Manhattan distance.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [7]:
labels = dbscan.labels_
np.unique(labels)

array([-1,  0,  1])

In [8]:
# Visualizar clusters en 2 dimensiones
df['Cluster 2D'] = labels

fig = px.scatter(
    df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    color='Cluster 2D',
    title='Clusters DBSCAN en 2D'
)   
fig.update_traces(marker=dict(size=10))
fig.show()

In [9]:
# Evaluar con silhouette
silhouette_avg = silhouette_score(X_scaled_2d, labels)
print(f'Silhouette Score: {silhouette_avg:.4f}')

Silhouette Score: 0.3504


In [10]:
# Ahora 3D
# Crear nuestra matriz X
X = df.drop(columns=['CustomerID', 'Genre'])

# Escalar nuestros datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled.shape

(200, 4)

In [11]:
dbscan_3d = DBSCAN(
    eps=0.5,
    min_samples=5,
    metric='euclidean'
)
dbscan_3d.fit(X_scaled)
dbscan_3d

,"eps eps: float, default=0.5The maximum distance between two samples for one to be consideredas in the neighborhood of the other. This is not a maximum boundon the distances of points within a cluster. This is the mostimportant DBSCAN parameter to choose appropriately for your data setand distance function. Smaller values generally lead to more clusters.",0.5
,"min_samples min_samples: int, default=5The number of samples (or total weight) in a neighborhood for a point tobe considered as a core point. This includes the point itself. If`min_samples` is set to a higher value, DBSCAN will find denser clusters,whereas if it is set to a lower value, the found clusters will be moresparse.",5
,"metric metric: str, or callable, default='euclidean'The metric to use when calculating distance between instances in afeature array. If metric is a string or callable, it must be one ofthe options allowed by :func:`sklearn.metrics.pairwise_distances` forits metric parameter.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors for DBSCAN... versionadded:: 0.17 metric *precomputed* to accept precomputed sparse matrix.",'euclidean'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function... versionadded:: 0.19",None
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'The algorithm to be used by the NearestNeighbors moduleto compute pointwise distances and find nearest neighbors.'auto' will attempt to decide the most appropriate algorithmbased on the values passed to :meth:`fit` method.See :class:`~sklearn.neighbors.NearestNeighbors` documentation fordetails.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or cKDTree. This can affect the speedof the construction and query, as well as the memory requiredto store the tree. The optimal value dependson the nature of the problem.",30
,"p p: float, default=NoneThe power of the Minkowski metric to be used to calculate distancebetween points. If None, then ``p=2`` (equivalent to the Euclideandistance). When p=1, this is equivalent to Manhattan distance.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [12]:
labels_3d = dbscan_3d.labels_
np.unique(labels_3d)

array([-1,  0,  1,  2,  3,  4,  5])

In [13]:
# Visualizar clusters en 3 dimensiones
df['Cluster 3D'] = labels_3d

fig = px.scatter_3d(
    df,
    x='Age',
    y='Annual Income (k$)',
    z='Spending Score (1-100)',
    color='Cluster 3D',
    title='Clusters DBSCAN en 3D'
)
fig.update_traces(marker=dict(size=5))
fig.show()

In [14]:
# Evaluar modelo
silhouette_avg_3d = silhouette_score(X_scaled, labels_3d)
print(f'Silhouette Score 3D: {silhouette_avg_3d:.4f}')

Silhouette Score 3D: 0.2138


# Mejorar modelo
Entrenar con 2 y 3 dimensiones, y diferentes combinaciones de paramétros, para obtener la mejor silueta.


In [15]:

# Búsqueda de hiperparámetros para 2D y 3D
eps_values = [0.3, 0.5, 0.7, 1.0, 1.5]
min_samples_values = [3, 5, 7, 10]

results = []

for dims, X_data in [('2D', X_scaled_2d), ('3D', X_scaled)]:
    for eps in eps_values:
        for min_samples in min_samples_values:
            lbls = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean').fit_predict(X_data)
            n_clusters = len(set(lbls)) - (1 if -1 in lbls else 0)
            n_noise = (lbls == -1).sum()

            # Silhouette requiere al menos 2 clusters y algún punto no-ruido
            if n_clusters >= 2:
                score = silhouette_score(X_data, lbls)
            else:
                score = None

            results.append({
                'Dimensiones': dims,
                'eps': eps,
                'min_samples': min_samples,
                'n_clusters': n_clusters,
                'n_noise': n_noise,
                'silhouette': score
            })

results_df = pd.DataFrame(results)
results_df_valid = results_df.dropna(subset=['silhouette'])
results_df_valid.sort_values('silhouette', ascending=False).head(10)

,Dimensiones,eps,min_samples,n_clusters,n_noise,silhouette
39,3D,1.5,10,2,8,0.447516
38,3D,1.5,7,2,8,0.447516
37,3D,1.5,5,2,8,0.447516
34,3D,1.0,7,2,9,0.439949
33,3D,1.0,5,2,9,0.439949
36,3D,1.5,3,4,0,0.436384
35,3D,1.0,10,2,10,0.433165
32,3D,1.0,3,3,5,0.426334
0,2D,0.3,3,9,14,0.413619
7,2D,0.5,10,4,21,0.406405


In [16]:

# Mejor combinación de parámetros
best = results_df_valid.loc[results_df_valid['silhouette'].idxmax()]

print(f"Mejor modelo:")
print(f"  Dimensiones : {best['Dimensiones']}")
print(f"  eps         : {best['eps']}")
print(f"  min_samples : {best['min_samples']}")
print(f"  Clusters    : {best['n_clusters']}")
print(f"  Ruido       : {best['n_noise']}")
print(f"  Silhouette  : {best['silhouette']:.4f}")

# Reentrenar con los mejores parámetros
X_best = X_scaled_2d if best['Dimensiones'] == '2D' else X_scaled
best_labels = DBSCAN(eps=best['eps'], min_samples=best['min_samples'], metric='euclidean').fit_predict(X_best)
df['Cluster Best'] = best_labels

titulo = f"Mejor DBSCAN ({best['Dimensiones']}) | eps={best['eps']}, min_samples={best['min_samples']}, Silhouette={best['silhouette']:.4f}"

if best['Dimensiones'] == '2D':
    fig = px.scatter(
        df,
        x='Annual Income (k$)',
        y='Spending Score (1-100)',
        color='Cluster Best',
        title=titulo
    )
    fig.update_traces(marker=dict(size=10))
else:
    fig = px.scatter_3d(
        df,
        x='Age',
        y='Annual Income (k$)',
        z='Spending Score (1-100)',
        color='Cluster Best',
        title=titulo
    )
    fig.update_traces(marker=dict(size=5))

fig.show()

Mejor modelo:
  Dimensiones : 3D
  eps         : 1.5
  min_samples : 5
  Clusters    : 2
  Ruido       : 8
  Silhouette  : 0.4475
